# EWMA VaR Evaluation ? 12/08/2026

This notebook evaluates the canonical EWMA VaR baseline against the existing Historical Simulation baseline on exactly the same target dates.

The notebook is an analysis artifact. It does not replace the common walk-forward runner or configuration artifacts owned elsewhere. Historical forecasts are read from the established canonical backtest file; only EWMA forecasts are rebuilt here for controlled comparison.

In [1]:
import numpy as np
import pandas as pd
from src.models.ewma_var import ewma_var_forecast

In [2]:
RETURNS_PATH = "data/processed/portfolio_returns.csv"
HISTORICAL_PATH = "data/processed/historical_var_backtest.csv"
ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95
DECAY = 0.94
EXPECTED_ROWS = 1387

## 1. Load canonical data and lock target dates

In [3]:
returns = pd.read_csv(RETURNS_PATH, parse_dates=["date"])
historical = pd.read_csv(HISTORICAL_PATH, parse_dates=["forecast_date", "target_date"])

assert len(historical) == EXPECTED_ROWS
assert not historical["target_date"].duplicated().any()
assert not historical.isna().any().any()

print(f"Portfolio return rows: {len(returns)}")
print(f"Historical rows: {len(historical)}")
print(f"Target start: {historical['target_date'].iloc[0].date()}")
print(f"Target end: {historical['target_date'].iloc[-1].date()}")

Portfolio return rows: 1637
Historical rows: 1387
Target start: 2020-12-31
Target end: 2026-07-28


## 2. Rebuild EWMA forecasts with no look-ahead

In [4]:
ewma_rows = []

for row in historical.itertuples(index=False):
    history = returns.loc[returns["date"] <= row.forecast_date, ["date", "portfolio_simple_return"]]
    target = returns.loc[returns["date"] == row.target_date, "portfolio_simple_return"]

    assert not history.empty
    assert history["date"].max() == row.forecast_date
    assert (history["date"] < row.target_date).all()
    assert len(target) == 1

    actual_return = float(target.iloc[0])
    assert abs(actual_return - float(row.target_return)) <= 1e-12

    forecast = ewma_var_forecast(history["portfolio_simple_return"].to_numpy(), alpha=ALPHA, decay=DECAY)
    ewma_rows.append({"forecast_date": row.forecast_date, "target_date": row.target_date, "observations": len(history), "quantile_return": forecast["quantile_return"], "ewma_var": forecast["var"], "target_return": actual_return, "violation": actual_return < forecast["quantile_return"], "method": "ewma"})

ewma = pd.DataFrame(ewma_rows)

In [5]:
assert len(ewma) == EXPECTED_ROWS
assert ewma["target_date"].equals(historical["target_date"])
assert not ewma["target_date"].duplicated().any()
assert not ewma.isna().any().any()
assert (ewma["ewma_var"] >= 0.0).all()
assert np.allclose(ewma["ewma_var"], np.maximum(0.0, -ewma["quantile_return"]), atol=1e-12, rtol=0.0)
assert np.array_equal(ewma["violation"].to_numpy(), (ewma["target_return"] < ewma["quantile_return"]).to_numpy())
assert int(ewma["violation"].sum()) == 73
assert abs(float(ewma["ewma_var"].min()) - 0.011438399048) <= 1e-12
assert abs(float(ewma["ewma_var"].max()) - 0.056660749124) <= 1e-12

print(f"Rows: {len(ewma)}")
print(f"Target start: {ewma['target_date'].iloc[0].date()}")
print(f"Target end: {ewma['target_date'].iloc[-1].date()}")
print(f"Min observations: {ewma['observations'].min()}")
print(f"Max observations: {ewma['observations'].max()}")
print(f"Min EWMA VaR: {ewma['ewma_var'].min():.12f}")
print(f"Max EWMA VaR: {ewma['ewma_var'].max():.12f}")
print(f"Violations: {int(ewma['violation'].sum())}")

Rows: 1387
Target start: 2020-12-31
Target end: 2026-07-28
Min observations: 250
Max observations: 1636
Min EWMA VaR: 0.011438399048
Max EWMA VaR: 0.056660749124
Violations: 73


## 3. Common-date Historical vs EWMA metrics

In [6]:
historical_eval = historical[["target_date", "quantile_return", "historical_var", "target_return", "violation"]].copy().rename(columns={"historical_var": "var"})
ewma_eval = ewma[["target_date", "quantile_return", "ewma_var", "target_return", "violation"]].copy().rename(columns={"ewma_var": "var"})

assert historical_eval["target_date"].equals(ewma_eval["target_date"])
assert np.allclose(historical_eval["target_return"], ewma_eval["target_return"], atol=1e-12, rtol=0.0)

def pinball_loss(y, q, alpha):
    error = y - q
    return float(np.mean(np.where(error >= 0.0, alpha * error, (alpha - 1.0) * error)))

def summarize(frame):
    violations = int(frame["violation"].sum())
    violation_rate = float(frame["violation"].mean())
    return {"forecasts": len(frame), "violations": violations, "violation_rate": violation_rate, "violation_rate_difference_pp": (violation_rate - ALPHA) * 100.0, "pinball_loss": pinball_loss(frame["target_return"].to_numpy(), frame["quantile_return"].to_numpy(), ALPHA), "average_var": float(frame["var"].mean()), "minimum_var": float(frame["var"].min()), "maximum_var": float(frame["var"].max())}

historical_metrics = summarize(historical_eval)
ewma_metrics = summarize(ewma_eval)

assert historical_metrics["forecasts"] == EXPECTED_ROWS
assert historical_metrics["violations"] == 75
assert abs(historical_metrics["pinball_loss"] - 0.002057488935) <= 1e-12
assert ewma_metrics["forecasts"] == EXPECTED_ROWS
assert ewma_metrics["violations"] == 73
assert abs(ewma_metrics["pinball_loss"] - 0.001962062254) <= 1e-12

for name, metrics in [("Historical", historical_metrics), ("EWMA", ewma_metrics)]:
    print(f"{name}: forecasts={metrics['forecasts']}, violations={metrics['violations']}, violation_rate={metrics['violation_rate']:.9%}, diff_from_5pct={metrics['violation_rate_difference_pp']:+.6f} pp, pinball_loss={metrics['pinball_loss']:.12f}, average_var={metrics['average_var']:.9%}, min_var={metrics['minimum_var']:.9%}, max_var={metrics['maximum_var']:.9%}")

print(f"EWMA - Historical violations: {ewma_metrics['violations'] - historical_metrics['violations']:+d}")
print(f"EWMA - Historical violation rate: {(ewma_metrics['violation_rate'] - historical_metrics['violation_rate']) * 100.0:+.6f} pp")
print(f"EWMA - Historical pinball loss: {ewma_metrics['pinball_loss'] - historical_metrics['pinball_loss']:+.12f}")
print(f"EWMA - Historical average VaR: {(ewma_metrics['average_var'] - historical_metrics['average_var']) * 100.0:+.6f} pp")

Historical: forecasts=1387, violations=75, violation_rate=5.407354001%, diff_from_5pct=+0.407354 pp, pinball_loss=0.002057488935, average_var=2.623147198%, min_var=1.708838523%, max_var=4.378556664%
EWMA: forecasts=1387, violations=73, violation_rate=5.263157895%, diff_from_5pct=+0.263158 pp, pinball_loss=0.001962062254, average_var=2.575468841%, min_var=1.143839905%, max_var=5.666074912%
EWMA - Historical violations: -2
EWMA - Historical violation rate: -0.144196 pp
EWMA - Historical pinball loss: -0.000095426681
EWMA - Historical average VaR: -0.047678 pp


## 4. Exception overlap

In [7]:
comparison = historical[["target_date", "target_return", "quantile_return", "historical_var", "violation"]].copy()
comparison = comparison.rename(columns={"quantile_return": "historical_quantile", "violation": "historical_violation"})
comparison = comparison.merge(ewma[["target_date", "quantile_return", "ewma_var", "violation"]].rename(columns={"quantile_return": "ewma_quantile", "violation": "ewma_violation"}), on="target_date", how="inner", validate="one_to_one")

common = comparison.loc[comparison["historical_violation"] & comparison["ewma_violation"]].copy()
historical_only = comparison.loc[comparison["historical_violation"] & ~comparison["ewma_violation"]].copy()
ewma_only = comparison.loc[~comparison["historical_violation"] & comparison["ewma_violation"]].copy()

assert int(comparison["historical_violation"].sum()) == 75
assert int(comparison["ewma_violation"].sum()) == 73
assert len(common) == 56
assert len(historical_only) == 19
assert len(ewma_only) == 17

common["historical_loss_excess"] = common["historical_quantile"] - common["target_return"]
common["ewma_loss_excess"] = common["ewma_quantile"] - common["target_return"]
common["max_loss_excess"] = common[["historical_loss_excess", "ewma_loss_excess"]].max(axis=1)

print(f"Historical exceptions: {int(comparison['historical_violation'].sum())}")
print(f"EWMA exceptions: {int(comparison['ewma_violation'].sum())}")
print(f"Common exceptions: {len(common)}")
print(f"Historical-only exceptions: {len(historical_only)}")
print(f"EWMA-only exceptions: {len(ewma_only)}")
print(f"Overlap / Historical: {len(common) / 75:.6%}")
print(f"Overlap / EWMA: {len(common) / 73:.6%}")

Historical exceptions: 75
EWMA exceptions: 73
Common exceptions: 56
Historical-only exceptions: 19
EWMA-only exceptions: 17
Overlap / Historical: 74.666667%
Overlap / EWMA: 76.712329%


In [8]:
print("Historical-only exceptions")
display(historical_only[["target_date", "target_return", "historical_quantile", "ewma_quantile"]])

print("EWMA-only exceptions")
display(ewma_only[["target_date", "target_return", "historical_quantile", "ewma_quantile"]])

print("Top 10 common exceptions")
display(common.nlargest(10, "max_loss_excess")[["target_date", "target_return", "historical_loss_excess", "ewma_loss_excess"]])

Historical-only exceptions


,target_date,target_return,historical_quantile,ewma_quantile
73,2021-04-22,-0.024475,-0.024062,-0.025871
75,2021-04-26,-0.026977,-0.024782,-0.029249
104,2021-06-08,-0.024962,-0.024782,-0.027015
128,2021-07-12,-0.029796,-0.024743,-0.036327
130,2021-07-14,-0.026669,-0.025002,-0.036124
158,2021-08-23,-0.028574,-0.025002,-0.029010
331,2022-05-06,-0.022535,-0.021813,-0.031026
362,2022-06-20,-0.039299,-0.026356,-0.041753
441,2022-10-11,-0.039076,-0.026789,-0.040117
459,2022-11-04,-0.040659,-0.035116,-0.042602


EWMA-only exceptions


,target_date,target_return,historical_quantile,ewma_quantile
220,2021-11-19,-0.017196,-0.024743,-0.015976
230,2021-12-03,-0.020514,-0.024743,-0.017938
231,2021-12-06,-0.022372,-0.024743,-0.019255
256,2022-01-11,-0.016059,-0.024743,-0.014852
295,2022-03-14,-0.020882,-0.021813,-0.019898
424,2022-09-16,-0.021727,-0.023092,-0.018132
619,2023-06-29,-0.021106,-0.030995,-0.015857
642,2023-08-01,-0.024973,-0.030995,-0.019709
678,2023-09-22,-0.026324,-0.035094,-0.023691
679,2023-09-25,-0.031010,-0.035094,-0.025300


Top 10 common exceptions


,target_date,target_return,historical_loss_excess,ewma_loss_excess
1059,2025-04-03,-0.069840,0.052047,0.052171
1061,2025-04-08,-0.069299,0.050871,0.036044
324,2022-04-25,-0.069485,0.047672,0.046187
1288,2026-03-09,-0.069402,0.044389,0.044487
124,2021-07-06,-0.064361,0.040882,0.041747
12,2021-01-19,-0.060883,0.023155,0.041480
435,2022-10-03,-0.060254,0.036162,0.038823
19,2021-01-28,-0.069552,0.033357,0.036926
332,2022-05-09,-0.058539,0.036217,0.027118
335,2022-05-12,-0.055981,0.033519,0.017163


## 5. Interpretation

Historical Simulation and EWMA identify a substantial common set of exception days, with 56 dates classified as violations by both methods. The overlap corresponds to approximately 74.67% of Historical exceptions and 76.71% of EWMA exceptions, while 19 dates are Historical-only exceptions and 17 dates are EWMA-only exceptions. This indicates that the two methods often detect the same severe losses but can produce materially different dynamic risk thresholds on individual dates.

On the common 1,387-date evaluation sample, EWMA has a violation rate of approximately 5.2632%, compared with 5.4074% for Historical Simulation, so the EWMA rate is descriptively closer to the nominal 5% level. EWMA also has a lower Pinball Loss, 0.001962062254 versus 0.002057488935. Its Average VaR is slightly lower, approximately 2.5755% versus 2.6231% for Historical Simulation.

The observed EWMA VaR range is wider: its minimum is approximately 1.1438% and its maximum approximately 5.6661%, compared with approximately 1.7088% and 4.3786% for Historical Simulation. This is consistent with the two approaches producing different time-varying risk thresholds.

These results are descriptive and do not establish statistical superiority of EWMA. Specific exception dates are not attributed to market events without separate supporting evidence. The canonical EWMA decay remains 0.94, and no parameter tuning is performed in this notebook.

## 6. Canonical checkpoints

In [9]:
print(f"Rows: {EXPECTED_ROWS}")
print(f"Historical violations: {historical_metrics['violations']}")
print(f"EWMA violations: {ewma_metrics['violations']}")
print(f"Historical violation rate: {historical_metrics['violation_rate']:.9%}")
print(f"EWMA violation rate: {ewma_metrics['violation_rate']:.9%}")
print(f"Historical pinball loss: {historical_metrics['pinball_loss']:.12f}")
print(f"EWMA pinball loss: {ewma_metrics['pinball_loss']:.12f}")
print(f"Historical average VaR: {historical_metrics['average_var']:.9%}")
print(f"EWMA average VaR: {ewma_metrics['average_var']:.9%}")
print(f"Common exceptions: {len(common)}")
print(f"Historical-only exceptions: {len(historical_only)}")
print(f"EWMA-only exceptions: {len(ewma_only)}")

Rows: 1387
Historical violations: 75
EWMA violations: 73
Historical violation rate: 5.407354001%
EWMA violation rate: 5.263157895%
Historical pinball loss: 0.002057488935
EWMA pinball loss: 0.001962062254
Historical average VaR: 2.623147198%
EWMA average VaR: 2.575468841%
Common exceptions: 56
Historical-only exceptions: 19
EWMA-only exceptions: 17
